# Foraging: does NCAP need a steering controller, or just training?

Trains **two** PPO runs on the `foraging` task, identical in every respect except one:

| run | controller | what's trained |
|---|---|---|
| `ncap_only_poc` | `None` | the circuit's own weights only |
| `learned_steering_poc` | `learned_steering` | the circuit's weights **and** a small steering MLP |

With `controller=None` the circuit is built with `include_turn_control=False`, so it has
no turn input to drive at all -- it can only swim, and the task's reward is the only
thing that changes. That is the paper's own setup, and the honest floor to read the
steered run against. `learned_steering` adds a tiny MLP mapping the egocentric direction
to the food onto one turn command, on top of the fixed NCAP turn primitive.

Two design choices make the steering learnable (both fixes for concrete failures earlier
in this project):
- **Warm start:** from random init PPO never even found the correct turn *sign* (~7%), so
  the controller is behaviour-cloned to the correct-sign hardcoded steerer first, and PPO
  refines from there.
- **Reward:** the swim-speed term is off (it rewards "swim fast" regardless of the food),
  leaving a dense **progress** reward (distance closed) plus an **eat bonus**. The same
  reward is used for both runs, so any gap between them is the controller's doing and not
  the shaping's.

The success metric below is deliberately **physics-only** -- the fraction of fresh
episodes where the head actually comes within `SUCCESS_DIST` of the food, read from
simulator state. Aggregate training reward has repeatedly lied in this environment
(a "great" score turned out to be almost entirely swim speed with no food-seeking under
it), so it is used for the learning curves and for nothing else.

Each run is ~1e5 steps, a couple of minutes. Re-running skips any run already on disk.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
SRC = str(Path.cwd().parent / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import matplotlib.pyplot as plt
from macrocircuits import ensure_tonic
ensure_tonic()

import torch, tonic, tonic.torch
from dm_control.mujoco import engine
from macrocircuits.models import ppo_swimmer_model
from macrocircuits.reflex_steering import make_steer_to_food_reflex
from macrocircuits.controllers import make_learned_steering
from macrocircuits.video import write_video
from IPython.display import Video

N_JOINTS = 5
SUCCESS_DIST = 0.15   # head within this of the food at some point => that episode found it
print('ready')

ready


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
def _get_physics(env):
    while env is not None:
        if hasattr(env, 'physics'):
            return env.physics
        inner = getattr(env, 'environment', None)
        if inner is not None and hasattr(inner, 'physics'):
            return inner.physics
        env = getattr(env, 'env', None)
    raise RuntimeError('no physics in env chain')


def _arena_frame(physics, distance=5.5):
    """Fixed top-down camera on the whole arena, so you see the worm traverse to food."""
    cam = engine.Camera(physics, height=480, width=640, camera_id=-1)
    rc = cam._render_camera
    rc.lookat[:] = [0.0, 0.0, 0.0]
    rc.distance = distance; rc.azimuth = 90; rc.elevation = -90
    frame = cam.render(); cam._scene.free()
    return frame


def reflex_policy(controller, seed=0):
    """A deterministic policy from a FRESH (untrained) NCAP circuit + the given controller.
    Returns (policy, env, physics); `policy(obs)` -> action, both batched (1, .)."""
    torch.manual_seed(seed)
    model = ppo_swimmer_model(n_joints=N_JOINTS, critic_sizes=(256, 256),
                              action_noise=0.1, controller=controller)
    env = tonic.environments.distribute(
        lambda: tonic.environments.ControlSuite('swimmer-foraging', time_feature=True))
    env.initialize(seed)
    actor = model.actor
    actor.initialize(observation_space=env.observation_space, action_space=env.action_space)
    def policy(obs):
        with torch.no_grad():
            out = actor(torch.as_tensor(obs, dtype=torch.float32))
            return (out.loc if hasattr(out, 'loc') else out).numpy()
    return policy, env, _get_physics(env.environments[0])


def trained_policy(path, seed=0):
    """Load a PPO-trained checkpoint from `path` and return (policy, env, physics)."""
    import argparse, os, yaml
    import macrocircuits.training as _t
    ns = dict(vars(_t))
    cfg = argparse.Namespace(**yaml.load(open(os.path.join(path, 'config.yaml')), Loader=yaml.FullLoader))
    try:
        if cfg.header: exec(cfg.header, ns)
    except Exception:
        pass
    agent = eval(cfg.agent, ns)
    env = tonic.environments.distribute(lambda: eval(cfg.environment, ns))
    env.initialize(seed)
    agent.initialize(observation_space=env.observation_space,
                     action_space=env.action_space, seed=seed)
    cp = os.path.join(path, 'checkpoints')
    ids = [int(n.split('.')[0][5:]) for n in os.listdir(cp) if n.startswith('step_')]
    agent.load(os.path.join(cp, f'step_{max(ids)}'))
    return (lambda obs: agent.test_step(obs, 0)), env, _get_physics(env.environments[0])


def success_rate(policy, env, n_episodes=30):
    """Fraction of fresh episodes whose head comes within SUCCESS_DIST of the food.
    Distance is read from infos['observations'] (the pre-reset transition obs)."""
    tgt = slice(N_JOINTS, N_JOINTS + 2)
    obs = env.start(); mind = np.inf; hits = []
    while len(hits) < n_episodes:
        obs, infos = env.step(policy(obs))
        mind = min(mind, float(np.linalg.norm(infos['observations'][0, tgt])))
        if infos['resets'][0]:
            hits.append(mind < SUCCESS_DIST); mind = np.inf
    return float(np.mean(hits))


def forage_episode(policy, env, physics, steps=1000, viz_food_size=0.10):
    """One episode; records overhead frames, head path, food positions, and the steps
    where a food was reached (it then respawns). viz_food_size only enlarges the food
    marker for the video -- the success metric uses the true tiny food."""
    obs = env.start()
    if viz_food_size:
        physics.named.model.geom_size['target', 0] = viz_food_size; physics.forward()
    frames, hx, hy, fx, fy, eats = [], [], [], [], [], []
    prev_food = None
    for i in range(steps):
        frames.append(_arena_frame(physics))
        nose = physics.named.data.geom_xpos['nose'][:2].copy()
        food = physics.named.data.geom_xpos['target'][:2].copy()
        hx.append(nose[0]); hy.append(nose[1]); fx.append(food[0]); fy.append(food[1])
        if prev_food is not None and np.linalg.norm(food - prev_food) > 1e-6:
            eats.append(i)
        prev_food = food
        obs, infos = env.step(policy(obs))
        if infos['resets'][0]:
            break
    return dict(frames=frames, hx=np.array(hx), hy=np.array(hy),
                fx=np.array(fx), fy=np.array(fy), eats=eats)


def show_video(frames, path, fps=30):
    """Write the frames to an mp4 (imageio) and embed the file directly. This plays
    reliably in VSCode / Jupyter, unlike matplotlib's to_html5_video, which often renders
    blank or hangs in the VSCode notebook viewer."""
    import os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_video(path, frames, fps=fps)
    return Video(path, embed=True, width=560)


def plot_path(ep, title):
    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(ep['hx'], ep['hy'], c=np.arange(len(ep['hx'])), cmap='viridis', s=6, label='head path')
    food_pts = np.unique(np.c_[ep['fx'], ep['fy']], axis=0)
    plt.scatter(food_pts[:, 0], food_pts[:, 1], marker='*', s=260, c='red', ec='k', label='food', zorder=5)
    if ep['eats']:
        plt.scatter(ep['hx'][ep['eats']], ep['hy'][ep['eats']], s=90, c='lime', ec='k', label='reached', zorder=6)
    plt.gca().set_aspect('equal'); plt.grid(alpha=0.3); plt.legend()
    plt.title(title); plt.xlabel('world x'); plt.ylabel('world y'); plt.show()

print('helpers ready')

helpers ready


## Train

Declares both runs and trains each (or reuses an existing checkpoint).

In [ ]:
import time, traceback
from macrocircuits.training import resolve_runs, run_config, run_path, is_trained, train

STEPS = int(1e5)   # bump for a longer run

# Held identical across both runs, so the only difference between them is the controller.
# envs.foraging also accepts alignment_reward_weight (credit for facing the food) and
# alignment_gated_progress_weight (progress scaled by how well it's facing the food) --
# both left at 0 here so this stays directly comparable with the existing
# learned_steering_poc checkpoint. Change them and both runs retrain, which is the point:
# is_trained() compares the environment string, so the reward is part of a run's identity.
TASK_KWARGS = dict(progress_reward_weight=10.0, eat_bonus=5.0)

RUNS = resolve_runs(
    [
        # NCAP alone -- no controller, so no turn input exists on the circuit at all.
        dict(controller=None, label='ncap_only_poc'),
        # NCAP + the learned steering MLP, whose weights PPO trains alongside the circuit's.
        dict(controller='learned_steering', label='learned_steering_poc'),
    ],
    defaults=dict(network='ncap', method='ppo', task='foraging',
                  task_kwargs=TASK_KWARGS, steps=STEPS),
)

PATHS, TRAIN_REPORT = {}, {}
for run in RUNS:
    label = run['label']
    agent_s, env_s, name, trainer_s = run_config(**run)
    path = run_path(**run)
    PATHS[label] = path
    t = time.time()
    try:
        if is_trained(path, agent_s, env_s, trainer_s):
            TRAIN_REPORT[label] = 'skipped (already trained)'
            print(f'{label:22} already trained -> {path}')
            continue
        print(f'{label:22} training {STEPS:,} steps ...')
        train(header='import tonic.torch', agent=agent_s, environment=env_s,
              name=name, trainer=trainer_s)
        TRAIN_REPORT[label] = f'trained in {(time.time() - t) / 60:.1f} min'
        print(f'{label:22} done -> {path}')
    # KeyboardInterrupt is a BaseException, so Interrupt Kernel still stops everything.
    except Exception as exc:
        TRAIN_REPORT[label] = f'FAILED: {type(exc).__name__}: {exc}'
        print(f'{label:22} FAILED -- continuing with the other run')
        traceback.print_exc()

print()
for label, outcome in TRAIN_REPORT.items():
    print(f'  {label:22} {outcome}')
PATHS

## Learning curves

Mean test-episode reward over training, both runs on one axis (from each run's `log.csv`).

Reward only -- read it as "did training converge", not as "did it forage". The success
plot below is the one that answers that.

In [ ]:
import pandas as pd, os

plt.figure(figsize=(6.5, 3.8))
for (label, run_dir), colour in zip(PATHS.items(), ['#bbb', '#2a9d8f']):
    log = pd.read_csv(os.path.join(run_dir, 'log.csv'))
    col = next(c for c in log.columns if 'episode_score/mean' in c)
    # Cumulative env steps. NOT 'train/epoch_steps', which is the constant epoch length
    # (== the trainer's epoch_steps, e.g. 20000) and is identical at every eval point --
    # plotting against it collapses the whole curve onto a single x. 'train/steps' is the
    # monotonic counter; it ends with '/steps' whereas epoch_steps ends with '_steps'.
    xcol = next(c for c in log.columns if c.endswith('/steps'))
    plt.plot(log[xcol], log[col], marker='o', color=colour, label=label)
plt.xlabel(xcol); plt.ylabel(col); plt.legend()
plt.title('training reward (NOT a foraging measure -- see below)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Evaluate: physics-only success

Both trained runs, each next to its own untrained starting point, so training's
contribution and the controller's contribution are separated:

- **NCAP only, untrained** -- the circuit as initialised, no steering. The floor.
- **NCAP only, PPO-trained** -- what the reward alone buys with no turn input to use.
- **+ steering, warm start, no RL** -- the behaviour-cloned controller before any training.
- **+ steering, PPO-trained** -- the full thing.

Success = the head came within `SUCCESS_DIST` of the food during the episode, measured
from simulator state, independent of whatever reward trained the checkpoint.

In [ ]:
ncap_only_pol, ncap_only_env, _ = trained_policy(PATHS['ncap_only_poc'])
steered_pol, steered_env, _ = trained_policy(PATHS['learned_steering_poc'])

rates = {
    'NCAP only\n(untrained)':        success_rate(*reflex_policy(None)[:2]),
    'NCAP only\n(PPO-trained)':      success_rate(ncap_only_pol, ncap_only_env),
    '+ steering\n(warm, no RL)':     success_rate(*reflex_policy(make_learned_steering(N_JOINTS))[:2]),
    '+ steering\n(PPO-trained)':     success_rate(steered_pol, steered_env),
}
for k, v in rates.items():
    print(f'{k.replace(chr(10), " "):28}: {v * 100:.0f}%')

plt.figure(figsize=(7, 4))
bars = plt.bar(range(len(rates)), [v * 100 for v in rates.values()],
               color=['#ddd', '#bbb', '#e9c46a', '#2a9d8f'], ec='k')
plt.xticks(range(len(rates)), list(rates))
plt.bar_label(bars, fmt='%.0f%%'); plt.ylabel('success (%)'); plt.ylim(0, 100)
plt.title(f'reaching food, 30 fresh episodes (within {SUCCESS_DIST})')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## Watch both trained runs forage

Same seed, same food placement -- the difference on screen is the controller.

The camera is a fixed overhead shot of the whole arena, deliberately: the model's
tracking cameras follow the worm's position but never rotate with its heading, so a turn
can read as the wrong way round purely from the camera. Nothing rotates here, so
direction on screen is real.

In [ ]:
SEED = 3
EPISODES = {}
for label in PATHS:
    pol, env, phys = trained_policy(PATHS[label], seed=SEED)
    EPISODES[label] = forage_episode(pol, env, phys, steps=1000)
    print(f'{label:22} food reached this episode: {len(EPISODES[label]["eats"])}')

show_video(EPISODES['learned_steering_poc']['frames'][::2],
           'output_videos/forage_trained_steering.mp4', fps=30)

In [ ]:
show_video(EPISODES['ncap_only_poc']['frames'][::2],
           'output_videos/forage_trained_ncap_only.mp4', fps=30)

In [ ]:
for label in PATHS:
    plot_path(EPISODES[label], f'{label} (trained): head path and food')